# M01 Superposition


## Toy 设定

我们把 4 个稀疏概念压进 2 维隐藏空间。即使隐藏平面发生重叠，重建依然可能很好。


In [ ]:
import torch
import matplotlib.pyplot as plt

torch.manual_seed(7)

num_features = 4
hidden_dim = 2
encoder = torch.nn.Linear(num_features, hidden_dim, bias=False)
decoder = torch.nn.Linear(hidden_dim, num_features, bias=False)
optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.05)

for step in range(500):
    active = (torch.rand(512, num_features) < 0.22).float()
    strengths = torch.rand(512, num_features)
    batch = active * strengths
    hidden = encoder(batch)
    recon = decoder(hidden)
    loss = torch.nn.functional.mse_loss(recon, batch) + 0.002 * hidden.abs().mean()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

samples = torch.eye(num_features)
hidden_samples = encoder(samples).detach()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(hidden_samples[:, 0], hidden_samples[:, 1], s=130, c=range(num_features), cmap="tab10")
for index, point in enumerate(hidden_samples):
    axes[0].annotate(f"f{index}", (point[0].item(), point[1].item()))
axes[0].set_title("Feature positions in 2D hidden space")
axes[0].axhline(0, color="0.8", linewidth=1)
axes[0].axvline(0, color="0.8", linewidth=1)

axes[1].imshow(decoder.weight.detach().T, cmap="coolwarm", aspect="auto")
axes[1].set_title("Decoder weights")
axes[1].set_xlabel("Hidden dimension")
axes[1].set_ylabel("Original feature")
plt.tight_layout()

print("Final loss:", float(loss.detach()))
print("Hidden representations:\n", hidden_samples)


## 小结

隐藏空间比概念集合更小，于是模型只能把多个概念塞进同一片区域里。这就是 superposition 的基本压力来源。
